In [2]:
%pip install dash

  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 3.8 MB/s eta 0:00:0000:0100:01
Using cached retrying-1.4.2-py3-none-any.whl (10 kB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# Load dataset
df = pd.read_csv("automobile_sales.csv")

# Create Dash application
app = Dash(__name__)

# Get available years
years = sorted(df["year"].unique())

# Dashboard layout
app.layout = html.Div([

    html.H1(
        "XYZAutomotives Sales Dashboard",
        style={"textAlign": "center"}
    ),

    html.Div([
        html.Label("Select Report Type:"),

        dcc.Dropdown(
            id="report_type",
            options=[
                {
                    "label": "Recession Report Statistics",
                    "value": "recession"
                },
                {
                    "label": "Yearly Report Statistics",
                    "value": "yearly"
                }
            ],
            value="recession",
            clearable=False
        )
    ], style={"width": "50%", "margin": "auto"}),

    html.Br(),

    html.Div([
        html.Label("Select Year:"),

        dcc.Dropdown(
            id="year_dropdown",
            options=[
                {
                    "label": str(year),
                    "value": year
                }
                for year in years
            ],
            value=years[0],
            clearable=False
        )
    ], style={"width": "50%", "margin": "auto"}),

    html.Br(),

    html.Div(
        id="output_container",
        className="output_container"
    ),

    dcc.Graph(id="output_graph")
])


# Callback
@app.callback(
    [
        Output("output_container", "children"),
        Output("output_graph", "figure")
    ],
    [
        Input("report_type", "value"),
        Input("year_dropdown", "value")
    ]
)
def update_dashboard(report_type, selected_year):

    # Recession Report
    if report_type == "recession":

        recession_data = df[df["recession"] == 1]

        yearly_sales = (
            recession_data
            .groupby("year")["sales_volume"]
            .sum()
            .reset_index()
        )

        fig = px.line(
            yearly_sales,
            x="year",
            y="sales_volume",
            markers=True,
            title="Automobile Sales During Recession"
        )

        title = "Recession Report Statistics"

    # Yearly Report
    else:

        yearly_data = df[df["year"] == selected_year]

        vehicle_sales = (
            yearly_data
            .groupby("vehicle_type")["sales_volume"]
            .sum()
            .reset_index()
        )

        fig = px.bar(
            vehicle_sales,
            x="vehicle_type",
            y="sales_volume",
            title=f"Automobile Sales by Vehicle Type - {selected_year}"
        )

        title = f"Yearly Report Statistics - {selected_year}"

    return html.H3(title), fig


# Run Dash application
if __name__ == "__main__":
    app.run(debug=True)